# Eksperimen 1: Perancangan Arsitektur Hibrid RT-DETR + ConvNeXt V2 + CBAM

**Judul Skripsi:**  
IMPLEMENTASI HIBRID RT-DETR DAN CONVNEXT V2 TERINTEGRASI CBAM UNTUK KLASIFIKASI DAN KUANTIFIKASI KUMPULAN BIJI FAMILI FABACEAE

**Mahasiswa Peneliti:** Refael Juari Siagian (NIM: 231402055)  
**Program Studi:** S-1 Teknologi Informasi, Fasilkom-TI, Universitas Sumatera Utara  
**Dosen Pembimbing:**  
1. Prof. Dr. Romi Fadillah Rahmat, B.Comp.Sc., M.Sc.  
2. Dr. Deden Sumirat Hidayat, M.Kom.

---  
## 1. Verifikasi Lingkungan & Akselerasi GPU (NVIDIA GeForce RTX 4070)
Cell ini memastikan PyTorch mendeteksi GPU NVIDIA dan framework Ultralytics siap digunakan.

In [ ]:
import sys
from pathlib import Path

# Tambahkan path ultralytics_repo langsung ke sys.path
repo_path = str(Path('ultralytics_repo').resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
import ultralytics

print("=" * 65)
print(f"Python Executable : {sys.executable}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Tersedia     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nama GPU          : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM GPU    : {vram_gb:.2f} GB")
print(f"Lokasi Ultralytics: {ultralytics.__file__}")
print("=" * 65)
ultralytics.checks()

---  
## 2. Pembuatan Modul Jaringan Kustom (CBAM & ConvNeXt V2)

Sesuai dengan rumusan metodologi skripsi:
1. **CBAM (Convolutional Block Attention Module):** Menggabungkan *Channel Attention* (interaksi antar-kanal fitur) dan *Spatial Attention* (fokus pada siluet batas antar-biji yang saling berhimpitan) untuk meredam *noise* latar belakang.
2. **ConvNeXt V2:** Mengintegrasikan **Global Response Normalization (GRN)** untuk mencegah kejenuhan fitur (*feature collapse*) serta menangkap tekstur morfologi halus biji Fabaceae.

Kode di bawah ini menggunakan perintah magic `%%writefile` untuk menulis modul secara otomatis ke `ultralytics/ultralytics/nn/modules/custom_modules.py`.

In [ ]:
%%writefile ultralytics_repo/ultralytics/nn/modules/custom_modules.py
# ====================================================================
# MODUL ARSITEKTUR KUSTOM SKRIPSI - REFAEL JUARI SIAGIAN (231402055)
# Implementasi Hibrid ConvNeXt V2 & CBAM terintegrasi RT-DETR
# ====================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------
# 1. MODUL CBAM (Convolutional Block Attention Module)
# --------------------------------------------------------------------

class ChannelAttention(nn.Module):
    """Channel Attention Module CBAM (Woo et al., 2018)."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced_ch = max(channels // reduction, 8)
        self.fc1 = nn.Conv2d(channels, reduced_ch, kernel_size=1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(reduced_ch, channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = F.adaptive_avg_pool2d(x, (1, 1))
        max_pool = F.adaptive_max_pool2d(x, (1, 1))
        avg_out = self.fc2(self.relu(self.fc1(avg_pool)))
        max_out = self.fc2(self.relu(self.fc1(max_pool)))
        scale = self.sigmoid(avg_out + max_out)
        return x * scale


class SpatialAttention(nn.Module):
    """Spatial Attention Module CBAM (Woo et al., 2018)."""
    def __init__(self, kernel_size=7):
        super().__init__()
        assert kernel_size in (3, 7), "Kernel size harus 3 atau 7"
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        scale = self.sigmoid(self.conv(concat))
        return x * scale


class CBAM(nn.Module):
    """CBAM terpadu (Channel Attention diikuti Spatial Attention)."""
    def __init__(self, c1, c2=None, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(c1, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


# --------------------------------------------------------------------
# 2. MODUL CONVNEXT V2 & GLOBAL RESPONSE NORMALIZATION (GRN)
# --------------------------------------------------------------------

class GRN(nn.Module):
    """Global Response Normalization (GRN) (Woo et al., 2023 - ConvNeXt V2).
    Mencegah feature collapse dan memperkuat pemodelan representasi spasial pada dense prediction."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.eps = eps

    def forward(self, x):
        Gx = torch.norm(x, p=2, dim=(2, 3), keepdim=True)
        Nx = Gx / (Gx.mean(dim=1, keepdim=True) + self.eps)
        return self.gamma * (x * Nx) + self.beta + x


class ConvNeXtV2_Block(nn.Module):
    """ConvNeXt V2 Block: 7x7 Depthwise Conv -> 1x1 Conv -> GELU -> GRN -> 1x1 Conv + Residual."""
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.BatchNorm2d(dim)
        self.pwconv1 = nn.Conv2d(dim, 4 * dim, kernel_size=1)
        self.act = nn.GELU()
        self.grn = GRN(4 * dim)
        self.pwconv2 = nn.Conv2d(4 * dim, dim, kernel_size=1)

    def forward(self, x):
        residual = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        return residual + x


class ConvNeXtV2_Stem(nn.Module):
    """Stem ConvNeXt V2: Patchify 4x4 konvolusi dengan stride 4 untuk mengecilkan dimensi spasial di awal."""
    def __init__(self, c1=3, c2=64):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size=4, stride=4)
        self.norm = nn.BatchNorm2d(c2)

    def forward(self, x):
        return self.norm(self.conv(x))


class ConvNeXtV2_Stage(nn.Module):
    """Stage ConvNeXt V2: Opsional downsampling 2x2 stride 2 diikuti kumpulan ConvNeXtV2 Blocks."""
    def __init__(self, c1, c2, depth=2, downsample=True):
        super().__init__()
        layers = []
        if downsample:
            layers.append(nn.Conv2d(c1, c2, kernel_size=2, stride=2))
            layers.append(nn.BatchNorm2d(c2))
            curr_dim = c2
        else:
            if c1 != c2:
                layers.append(nn.Conv2d(c1, c2, kernel_size=1))
                layers.append(nn.BatchNorm2d(c2))
                curr_dim = c2
            else:
                curr_dim = c1

        for _ in range(depth):
            layers.append(ConvNeXtV2_Block(curr_dim))
        self.stage = nn.Sequential(*layers)

    def forward(self, x):
        return self.stage(x)


---  
## 3. Pendaftaran Modul Baru ke Sistem Ultralytics
Cell ini secara otomatis mendaftarkan `CBAM`, `ConvNeXtV2_Stem`, dan `ConvNeXtV2_Stage` ke `nn/modules/__init__.py` serta menyematkan parsing parameternya ke dalam fungsi `parse_model()` pada `nn/tasks.py`.

In [ ]:
import os

modules_init_path = "ultralytics_repo/ultralytics/nn/modules/__init__.py"
tasks_path = "ultralytics_repo/ultralytics/nn/tasks.py"

# 1. Registrasi ke __init__.py
with open(modules_init_path, "r", encoding="utf-8") as f:
    init_content = f.read()

custom_export = (
    "\n# Custom Modules Refael Juari Siagian (231402055)\n"
    "from .custom_modules import CBAM, ConvNeXtV2_Stem, ConvNeXtV2_Stage, ConvNeXtV2_Block, GRN\n"
)

if "from .custom_modules import" not in init_content:
    with open(modules_init_path, "a", encoding="utf-8") as f:
        f.write(custom_export)
    print("✅ Berhasil mendaftarkan modul ke nn/modules/__init__.py")
else:
    print("ℹ️ Modul sudah terdaftar sebelumnya di nn/modules/__init__.py")

# 2. Registrasi ke tasks.py
with open(tasks_path, "r", encoding="utf-8") as f:
    tasks_content = f.read()

# Tambahkan import kustom jika belum ada
if "from ultralytics.nn.modules.custom_modules import" not in tasks_content:
    import_stmt = (
        "from ultralytics.nn.modules.custom_modules import (\n"
        "    CBAM, ConvNeXtV2_Stem, ConvNeXtV2_Stage, ConvNeXtV2_Block, GRN\n"
        ")\n"
    )
    tasks_content = import_stmt + tasks_content
    print("✅ Berhasil menambahkan import kustom ke tasks.py")

# Sisipkan handler parse_model
parse_hook = """        elif m is CBAM:
            c1 = ch[f]
            c2 = c1
            args = [c1, *args]
        elif m is ConvNeXtV2_Stem:
            c1 = ch[f]
            c2 = args[0]
            args = [c1, *args]
        elif m is ConvNeXtV2_Stage:
            c1 = ch[f]
            c2 = args[0]
            args = [c1, *args]
"""

if "elif m is ConvNeXtV2_Stage:" not in tasks_content:
    target_str = "        elif m is CBLinear:"
    if target_str in tasks_content:
        tasks_content = tasks_content.replace(target_str, parse_hook + target_str)
        with open(tasks_path, "w", encoding="utf-8") as f:
            f.write(tasks_content)
        print("✅ Berhasil menyisipkan handler parsing modul ke fungsi parse_model() tasks.py")
    else:
        print("⚠️ String target parsing tidak ditemukan, mohon periksa posisi parse_model()")
else:
    print("ℹ️ Handler parsing sudah terpasang di tasks.py")
print("Selesai! Ultralytics siap membaca arsitektur kustom YAML.")

---  
## 4. Konfigurasi Arsitektur YAML (`configs/rtdetr_convnextv2_cbam.yaml`)

Berikut adalah konfigurasi utuh arsitektur hibrid skripsi Anda:
- **Backbone:** Menggunakan hierarki ConvNeXt V2 (Stem -> Stage 0 -> Stage 1 [P3] -> Stage 2 [P4] -> Stage 3 [P5]) dengan modul **CBAM terintegrasi** di setiap output skala P3, P4, dan P5.
- **Head:** Menggunakan Hybrid Encoder (AIFI dan CCFM) untuk fusi multi-skala berkecepatan tinggi, diakhiri dengan **RTDETRDecoder** bebas NMS (*bipartite matching*).

In [ ]:
import os
os.makedirs("configs", exist_ok=True)

In [ ]:
%%writefile configs/rtdetr_convnextv2_cbam.yaml
# ====================================================================
# ARSITEKTUR HIBRID RT-DETR + CONVNEXT V2 + CBAM
# Riset Skripsi S-1 TI Fasilkom-TI USU: Refael Juari Siagian (231402055)
# ====================================================================

nc: 80 # Jumlah kelas (akan otomatis disesuaikan dengan dataset biji Fabaceae)
scales:
  l: [1.00, 1.00, 1024]

# --------------------------------------------------------------------
# 1. BACKBONE: ConvNeXt V2 terintegrasi CBAM
# --------------------------------------------------------------------
backbone:
  # [from, repeats, module, args]
  - [-1, 1, ConvNeXtV2_Stem, [64]]                    # 0: P2/4 (64 saluran)
  - [-1, 1, ConvNeXtV2_Stage, [64, 2, False]]          # 1: Stage 0 (64 saluran, stride 1)
  - [-1, 1, ConvNeXtV2_Stage, [128, 2, True]]          # 2: Stage 1 -> P3/8 (128 saluran, stride 2)
  - [-1, 1, CBAM, []]                                  # 3: Integrasi CBAM pada P3 (128 saluran)
  - [-1, 1, ConvNeXtV2_Stage, [256, 2, True]]          # 4: Stage 2 -> P4/16 (256 saluran, stride 2)
  - [-1, 1, CBAM, []]                                  # 5: Integrasi CBAM pada P4 (256 saluran)
  - [-1, 1, ConvNeXtV2_Stage, [512, 2, True]]          # 6: Stage 3 -> P5/32 (512 saluran, stride 2)
  - [-1, 1, CBAM, []]                                  # 7: Integrasi CBAM pada P5 (512 saluran)

# --------------------------------------------------------------------
# 2. HEAD: Hybrid Encoder (AIFI + CCFM) & RT-DETR Transformer Decoder
# --------------------------------------------------------------------
head:
  - [-1, 1, Conv, [256, 1, 1, None, 1, 1, False]]      # 8: Proyeksi P5 ke 256 saluran
  - [-1, 1, AIFI, [1024, 8]]                           # 9: Intra-scale Feature Interaction (AIFI)
  - [-1, 1, Conv, [256, 1, 1]]                         # 10: Y5

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]          # 11: Upsample Y5 ke skala P4
  - [5, 1, Conv, [256, 1, 1, None, 1, 1, False]]       # 12: Proyeksi CBAM-P4 (layer 5) ke 256 saluran
  - [[-2, -1], 1, Concat, [1]]                         # 13: Fusi fitur lateral skala P4
  - [-1, 3, RepC3, [256]]                              # 14: Cross-scale aggregation
  - [-1, 1, Conv, [256, 1, 1]]                         # 15: Y4

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]          # 16: Upsample Y4 ke skala P3
  - [3, 1, Conv, [256, 1, 1, None, 1, 1, False]]       # 17: Proyeksi CBAM-P3 (layer 3) ke 256 saluran
  - [[-2, -1], 1, Concat, [1]]                         # 18: Fusi fitur lateral skala P3
  - [-1, 3, RepC3, [256]]                              # 19: X3 (Output P3 terfusi)

  - [-1, 1, Conv, [256, 3, 2]]                         # 20: Downsample X3 ke skala P4
  - [[-1, 15], 1, Concat, [1]]                         # 21: Konkatenasi dengan Y4 (layer 15)
  - [-1, 3, RepC3, [256]]                              # 22: F4 (Output P4 terfusi)

  - [-1, 1, Conv, [256, 3, 2]]                         # 23: Downsample F4 ke skala P5
  - [[-1, 10], 1, Concat, [1]]                         # 24: Konkatenasi dengan Y5 (layer 10)
  - [-1, 3, RepC3, [256]]                              # 25: F5 (Output P5 terfusi)

  - [[19, 22, 25], 1, RTDETRDecoder, [nc]]             # 26: RTDETRDecoder(P3, P4, P5) Bebas-NMS


---  
## 5. Pengujian Kering (*Dry-Run*) di GPU RTX 4070

Cell ini melakukan validasi menyeluruh:
1. Membangun model PyTorch dari file konfigurasi YAML kustom di atas.
2. Memindahkan bobot model ke **NVIDIA GeForce RTX 4070**.
3. Membuat *dummy tensor* berukuran `(1, 3, 640, 640)` (simulasi 1 gambar RGB 640x640).
4. Menjalankan *forward pass* inferensi untuk membuktikan seluruh layer (ConvNeXt V2, GRN, CBAM, AIFI, dan RTDETRDecoder) terhubung sempurna tanpa *dimension mismatch*.

In [ ]:
import torch
import time
from ultralytics import RTDETR

# 1. Instansiasi model hibrid
print("Memuat dan menyusun arsitektur RT-DETR + ConvNeXt V2 + CBAM...")
model = RTDETR("configs/rtdetr_convnextv2_cbam.yaml")

# 2. Kirim model ke GPU RTX 4070
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
model.model.eval()

# 3. Buat dummy image tensor (Batch=1, Channels=3, Height=640, Width=640)
dummy_input = torch.randn(1, 3, 640, 640, device=device)
print(f"Tensor input disiapkan: {dummy_input.shape} di {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 4. Jalankan forward pass inferensi
print("Menjalankan forward pass...")
t_start = time.time()
with torch.no_grad():
    output = model(dummy_input)
t_end = time.time()

elapsed_ms = (t_end - t_start) * 1000
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print("=" * 65)
print("🎉 PENGUJIAN DRY-RUN BERHASIL 100% TANPA ERROR!")
print(f"Total Parameter Model       : {total_params:,} ({total_params / 1e6:.2f} M)")
print(f"Trainable Parameters        : {trainable_params:,}")
print(f"Waktu Komputasi Inferensi   : {elapsed_ms:.2f} ms")
print(f"Perangkat Komputasi Aktif   : {device}")
print("=" * 65)
print("Arsitektur hibrid skripsi Anda siap untuk dilatih pada dataset biji Fabaceae!")